# Highway OCD Benchmark — Quickstart

Compare overlapping community detection algorithms (Highway + baselines) on the same
graphs and metrics, and plug in **your own** algorithm with one line.

Metric panel: `extended_modularity` (Q_ov, unsupervised) + `fri`, `cover_sim_czekanowski`
(Dice), `fstar_wo` (F*), `onmi` (ONMI) when ground truth is available.

In [ ]:
import sys, pickle
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make `benchmark` importable

from benchmark import compare_algorithms, register_algorithm, available_algorithms
print('built-in algorithms:', available_algorithms())

## 1. Run the benchmark on a synthetic graph (with ground truth)

The shipped LFR / ABCD+o² graphs carry their ground-truth cover in the node attribute
`communities`, so all five metrics are computed automatically. The Highway C++ backend is
compiled on first use (needs `cmake` + a C++17 compiler).

In [ ]:
G = pickle.load(open('../data/synthetic/lfr_N200/graph.gpickle', 'rb'))
print(f'LFR N200: n={G.number_of_nodes()} m={G.number_of_edges()}')

df = compare_algorithms(G, algos=['highway', 'slpa', 'demon', 'lfm'])
df

## 2. Plug in your own algorithm

Your algorithm just needs the contract **`fn(G) -> list of communities`** (each community a
list of node labels; communities may overlap). Register it once and it appears in the table.

In [ ]:
import networkx as nx

def my_algorithm(G):
    # replace with your real method; this toy returns connected components
    return [list(c) for c in nx.connected_components(G)]

register_algorithm('my_algorithm', my_algorithm, overwrite=True)

compare_algorithms(G, algos=['highway', 'lfm', 'my_algorithm'])

## 3. Overlapping ground truth (ABCD+o²)

ABCD+o² nodes can belong to several communities — the overlapping metrics handle this.

In [ ]:
G2 = pickle.load(open('../data/synthetic/abcdo2_N200/graph.gpickle', 'rb'))
compare_algorithms(G2, algos=['highway', 'slpa', 'demon', 'lfm', 'my_algorithm'])